# Vérification d'affirmations scientifiques — SciFact

L'étage qui manquait : le moteur ne rend plus une liste de documents, il **décide**
si une affirmation est étayée ou contredite, **en citant les phrases exactes** qui
fondent la décision.

## Repères publiés — Wadden et al., EMNLP 2020, table 7 (jeu **dev**)

| Système, régime ouvert | Phrase, sél.+étiq. F1 | Abstract, étiq.+justif. F1 |
|---|---|---|
| Zéro-shot (entraîné sur FEVER) | 28,4 | 38,4 |
| **VeriSci** | **42,6** | **48,5** |
| VeriSci, abstracts fournis | 60,6 | 72,5 |
| Justifications fournies | 79,9 | 83,0 |

## Ce qui est déjà mesuré

Le plafond imposé par la récupération, avec **BM25 seul** :

| k | abstracts de preuve retrouvés | F1 d'un vérificateur parfait |
|---|---|---|
| 3 | 81,3 % | 89,7 |
| 10 | 89,5 % | 94,4 |
| 100 | 97,1 % | 98,5 |

VeriSci perdait 24 points entre le régime ouvert (48,5) et les abstracts fournis
(72,5) : c'était le coût de leur récupération TF-IDF. Ici le plafond est à 89,7
dès le top-3. **La récupération n'est plus le goulot ; tout se joue dans la
vérification.**

## Méthode

Un modèle d'inférence textuelle confronte l'affirmation à chaque phrase des
abstracts retrouvés. Ses trois sorties correspondent aux étiquettes de SciFact :
`entailment` → SUPPORT, `contradiction` → CONTRADICT, `neutral` → rien.

Aucun entraînement : le modèle est pré-entraîné sur MNLI, FEVER et ANLI.
**Le seuil est réglé sur `train`, jamais sur `dev`** — SciFact fournit 809
affirmations annotées pour cela.

L'évaluation utilise le **code officiel** d'AllenAI, validé en local : nourri de
la vérité terrain, il rend 1,0 sur les quatre métriques.

## 1. GPU et dépendances

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() or "AUCUN GPU")

In [ ]:
!pip -q install transformers torch nltk pandas 2>&1 | tail -2
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| cuda", torch.cuda.is_available())

## 2. Données et code d'évaluation officiel

In [ ]:
import json, os, tarfile, zipfile, urllib.request, numpy as np

# corpus de recherche (BEIR) et donnees de verification (AllenAI)
if not os.path.exists("scifact"):
    urllib.request.urlretrieve(
        "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip",
        "scifact.zip")
    zipfile.ZipFile("scifact.zip").extractall(".")

if not os.path.exists("data/claims_dev.jsonl"):
    urllib.request.urlretrieve(
        "https://scifact.s3-us-west-2.amazonaws.com/release/latest/data.tar.gz",
        "verif.tar.gz")
    tarfile.open("verif.tar.gz").extractall(".")

# code d'evaluation officiel
os.makedirs("evaluate/lib", exist_ok=True)
B = "https://raw.githubusercontent.com/allenai/scifact/master/verisci/evaluate"
for f in ["pipeline.py"]:
    urllib.request.urlretrieve(f"{B}/{f}", f"evaluate/{f}")
for f in ["__init__.py", "data.py", "metrics.py"]:
    urllib.request.urlretrieve(f"{B}/lib/{f}", f"evaluate/lib/{f}")

corpus = {str(json.loads(l)["doc_id"]): json.loads(l)
          for l in open("data/corpus.jsonl", encoding="utf-8")}
train = [json.loads(l) for l in open("data/claims_train.jsonl", encoding="utf-8")]
dev   = [json.loads(l) for l in open("data/claims_dev.jsonl", encoding="utf-8")]

print(f"corpus {len(corpus)} abstracts | train {len(train)} | dev {len(dev)}")
assert len(corpus) == 5183 and len(dev) == 300
print("contrôle données conforme")

## 3. Récupération — BM25

Même implémentation qu'en local, contrôlée à 0,6756 nDCG@10 contre 0,665 publié.

In [ ]:
import re
from collections import Counter
from nltk.stem.porter import PorterStemmer

ARRET = set('''a an and are as at be but by for if in into is it no not of on or
such that the their then there these they this to was will with'''.split())
_r, _cache = PorterStemmer(), {}

def normaliser(t):
    out = []
    for m in re.findall(r"[a-z0-9]+", t.lower()):
        if m in ARRET: continue
        s = _cache.get(m)
        if s is None:
            s = _r.stem(m); _cache[m] = s
        out.append(s)
    return out

class BM25:
    def __init__(self, docs, k1=0.9, b=0.4):
        self.k1, self.b = k1, b
        jetons = [normaliser(d) for d in docs]
        self.n = len(jetons)
        self.longueurs = np.array([len(d) for d in jetons], dtype=np.float32)
        self.moyenne = float(self.longueurs.mean())
        brut = {}
        for i, doc in enumerate(jetons):
            for t, f in Counter(doc).items():
                brut.setdefault(t, []).append((i, f))
        self.index = {}
        for t, post in brut.items():
            idx = np.array([p[0] for p in post], dtype=np.int32)
            frq = np.array([p[1] for p in post], dtype=np.float32)
            df = len(post)
            self.index[t] = (idx, frq, float(np.log(1 + (self.n - df + .5) / (df + .5))))
    def scores(self, q):
        s = np.zeros(self.n, dtype=np.float32)
        for t in normaliser(q):
            e = self.index.get(t)
            if e is None: continue
            idx, frq, idf = e
            norme = 1 - self.b + self.b * self.longueurs[idx] / self.moyenne
            s[idx] += idf * (frq * (self.k1 + 1)) / (frq + self.k1 * norme)
        return s

doc_ids = list(corpus)
textes_docs = [f"{corpus[d]['title']} {' '.join(corpus[d]['abstract'])}" for d in doc_ids]
lex = BM25(textes_docs)

def recuperer(claims, k=3):
    out = {}
    for c in claims:
        s = lex.scores(c["claim"])
        top = np.argpartition(-s, k)[:k]
        out[c["id"]] = [doc_ids[i] for i in top[np.argsort(-s[top])]]
    return out

K_ABSTRACTS = 3
rec_train = recuperer(train, K_ABSTRACTS)
rec_dev   = recuperer(dev,   K_ABSTRACTS)

# controle : part des abstracts de preuve effectivement retrouves
def couverture(claims, rec):
    vp = att = 0
    for c in claims:
        if not c.get("evidence"): continue
        vp += len(set(c["evidence"]) & set(rec[c["id"]]))
        att += len(c["evidence"])
    return vp / att

print(f"couverture train {couverture(train, rec_train):.3f} | "
      f"dev {couverture(dev, rec_dev):.3f}   (attendu ~0.81 sur dev au top-3)")

## 4. Modèle d'inférence textuelle

`premise` = la phrase de l'abstract, `hypothesis` = l'affirmation. On lit
`id2label` dans la configuration du modèle plutôt que de supposer l'ordre des
classes — une inversion silencieuse fausserait tout.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

NLI = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
tok = AutoTokenizer.from_pretrained(NLI)
nli = AutoModelForSequenceClassification.from_pretrained(NLI).to("cuda").eval().half()

id2label = {int(k): v.lower() for k, v in nli.config.id2label.items()}
print("classes du modèle :", id2label)
I_POUR    = next(i for i, v in id2label.items() if v.startswith("entail"))
I_CONTRE  = next(i for i, v in id2label.items() if v.startswith("contradic"))
I_NEUTRE  = next(i for i, v in id2label.items() if v.startswith("neutral"))
print(f"entailment={I_POUR}  contradiction={I_CONTRE}  neutral={I_NEUTRE}")

@torch.no_grad()
def probabilites(paires, lot=64):
    out = []
    for i in range(0, len(paires), lot):
        p = paires[i:i + lot]
        e = tok([x[0] for x in p], [x[1] for x in p], padding=True,
                truncation=True, max_length=256, return_tensors="pt").to("cuda")
        out.append(torch.softmax(nli(**e).logits.float(), dim=-1).cpu().numpy())
    return np.vstack(out) if out else np.zeros((0, 3), dtype=np.float32)

## 5. Scorer toutes les phrases des abstracts retrouvés

In [ ]:
import time

def scorer(claims, rec, nom=""):
    # (id_affirmation, doc) -> matrice (n_phrases, 3)
    paires, reperes = [], []
    for c in claims:
        for doc in rec[c["id"]]:
            phrases = corpus[doc]["abstract"]
            reperes.append((c["id"], doc, len(paires), len(paires) + len(phrases)))
            paires.extend((ph, c["claim"]) for ph in phrases)
    print(f"{nom} : {len(paires)} paires phrase × affirmation")
    t0 = time.time()
    P = probabilites(paires)
    print(f"  {time.time()-t0:.0f} s")
    return {(cid, doc): P[a:b] for cid, doc, a, b in reperes}

probas_train = scorer(train, rec_train, "train")
probas_dev   = scorer(dev,   rec_dev,   "dev")

## 6. Décision, et réglage du seuil **sur `train`**

Le seuil décide quelles phrases sont assez convaincantes pour être citées et
pour porter une étiquette. Il est balayé sur `train` uniquement ; `dev` ne sert
qu'à rapporter le résultat final.

In [ ]:
MAX_PHRASES = 3   # plafond du code d'evaluation officiel

def decider(P, seuil):
    if len(P) == 0:
        return None
    pour, contre, neutre = P[:, I_POUR], P[:, I_CONTRE], P[:, I_NEUTRE]
    if max(pour.max(), contre.max()) < seuil:
        return None
    if pour.max() >= contre.max():
        etiquette, scores = "SUPPORT", pour
    else:
        etiquette, scores = "CONTRADICT", contre
    retenues = np.where(scores >= seuil)[0]
    if len(retenues) == 0:
        retenues = np.array([int(scores.argmax())])
    retenues = retenues[np.argsort(-scores[retenues])][:MAX_PHRASES]
    return etiquette, sorted(int(i) for i in retenues)

def predire(claims, rec, probas, seuil):
    sortie = []
    for c in claims:
        preuve = {}
        for doc in rec[c["id"]]:
            d = decider(probas[(c["id"], doc)], seuil)
            if d is not None:
                preuve[str(doc)] = {"label": d[0], "sentences": d[1]}
        sortie.append({"id": c["id"], "evidence": preuve})
    return sortie

def evaluer_officiel(predictions, gold_path, nom=""):
    with open("pred.jsonl", "w") as f:
        for p in predictions:
            f.write(json.dumps(p) + "\n")
    r = subprocess.run(
        ["python", "pipeline.py", "--gold", f"../{gold_path}",
         "--corpus", "../data/corpus.jsonl", "--prediction", "../pred.jsonl",
         "--output", "../metriques.json"],
        cwd="evaluate", capture_output=True, text=True)
    if not os.path.exists("metriques.json"):
        print(r.stdout[-1500:], r.stderr[-1500:])
        raise RuntimeError("l'évaluateur officiel a échoué")
    m = json.load(open("metriques.json"))
    os.remove("metriques.json")
    return m

print(f"{'seuil':>7}{'phrase sél.+étiq.':>20}{'abstract étiq.+justif.':>26}")
print("-" * 55)
meilleur = (None, -1.0)
for seuil in (0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 0.95):
    m = evaluer_officiel(predire(train, rec_train, probas_train, seuil),
                         "data/claims_train.jsonl")
    ph = m["sentence_label"]["f1"] * 100
    ab = m["abstract_rationalized"]["f1"] * 100
    if ab > meilleur[1]:
        meilleur = (seuil, ab)
    print(f"{seuil:>7.2f}{ph:>20.1f}{ab:>26.1f}")

SEUIL = meilleur[0]
print(f"\nseuil retenu sur train : {SEUIL}")

## 7. Résultat sur `dev`, avec le seuil réglé sur `train`

In [ ]:
m = evaluer_officiel(predire(dev, rec_dev, probas_dev, SEUIL), "data/claims_dev.jsonl")

REPERES = {
    "Zéro-shot (FEVER), 2020":   {"phrase": 28.4, "abstract": 38.4},
    "VeriSci, régime ouvert":    {"phrase": 42.6, "abstract": 48.5},
    "VeriSci, abstracts fournis":{"phrase": 60.6, "abstract": 72.5},
    "Justifications fournies":   {"phrase": 79.9, "abstract": 83.0},
}

ph = m["sentence_label"]["f1"] * 100
ab = m["abstract_rationalized"]["f1"] * 100

print(f"{'système':<34}{'phrase':>10}{'abstract':>11}")
print("-" * 55)
for nom, v in REPERES.items():
    print(f"{nom:<34}{v['phrase']:>10.1f}{v['abstract']:>11.1f}")
print("-" * 55)
print(f"{'ce système (zéro-shot)':<34}{ph:>10.1f}{ab:>11.1f}")
print("-" * 55)
print(f"{'plafond de la récupération':<34}{'':>10}{89.7:>11.1f}")

print()
for cle, libelle in [("sentence_selection", "phrase, sélection seule"),
                     ("sentence_label",     "phrase, sélection + étiquette"),
                     ("abstract_label_only","abstract, étiquette seule"),
                     ("abstract_rationalized","abstract, étiquette + justification")]:
    d = m[cle]
    print(f"  {libelle:<38} P {d['precision']*100:>5.1f}  R {d['recall']*100:>5.1f}  F1 {d['f1']*100:>5.1f}")

json.dump({"seuil_regle_sur_train": SEUIL, "modele": NLI,
           "k_abstracts": K_ABSTRACTS, "metriques_dev": m, "reperes": REPERES},
          open("resultats_verification.json", "w"), indent=2)
print("\nécrit : resultats_verification.json")

## 8. Ce que le système produit — l'ancrage, en clair

Une décision ne vaut que si on peut remonter à ce qui la fonde. Voici quelques
affirmations avec la décision rendue et les phrases exactes citées.

In [ ]:
predictions = {p["id"]: p for p in predire(dev, rec_dev, probas_dev, SEUIL)}

montres = 0
for c in dev:
    p = predictions[c["id"]]
    if not p["evidence"] or montres >= 3:
        continue
    montres += 1
    print("=" * 78)
    print(f"AFFIRMATION  {c['claim']}")
    for doc, d in p["evidence"].items():
        or_ = c.get("evidence", {}).get(doc)
        juste = "✓" if or_ and or_[0]["label"] == d["label"] else "✗"
        print(f"\n  {juste} {d['label']}  —  abstract {doc}")
        print(f"    « {corpus[doc]['title']} »")
        for i in d["sentences"]:
            print(f"      [phrase {i}] {corpus[doc]['abstract'][i].strip()}")
        if or_:
            print(f"    or : {or_[0]['label']}, phrases "
                  f"{sorted({s for g in or_ for s in g['sentences']})}")
        else:
            print("    or : cet abstract n'est pas une preuve (faux positif)")

## 9. Récupérer

In [ ]:
from google.colab import files
files.download("resultats_verification.json")